# Metadata extraction from rental agreements
Extracts **Agreement Value, Start Date, End Date, Renewal Notice (days), Party One, Party Two** from `.docx` and scanned `.png` files using an LLM (no regex / rule-based logic).

Run the cells top to bottom. Documents that were already processed are read from `outputs/cache/`, so re-running costs no API calls.

In [1]:
import os, sys
from pathlib import Path

# make the notebook work whether it is started from the project root or from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from src.run import predict_folder
from src.evaluate import load_labels, per_field_recall, mismatches

ROOT = Path.cwd()
labels = load_labels(ROOT / "data" / "train.csv", ROOT / "data" / "test.csv")
pd.set_option("display.width", 200)
print("Project root:", ROOT)

Project root: c:\Users\Darshan\USEREADY TASKS\assignment-1


## 1. Predictions on the test set

In [2]:
pred_test = predict_folder(ROOT / "data" / "test", only=set(labels.index))
(ROOT / "outputs").mkdir(exist_ok=True)
pred_test.to_csv(ROOT / "outputs" / "predictions_test.csv")
pred_test

Cached     156155545-Rental-Agreement-Kns-Home.pdf.docx
Cached     228094620-Rental-Agreement.pdf.docx
Cached     24158401-Rental-Agreement.png
Cached     95980236-Rental-Agreement.png


,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
File Name,,,,,,
156155545-Rental-Agreement-Kns-Home,12000,15.12.2012,14.11.2013,30,V.K.NATARAJ,SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.
228094620-Rental-Agreement,15000,07.07.2013,06.06.2014,30,KAPIL MEHROTRA,B.Kishore
24158401-Rental-Agreement,12000,01.04.2008,31.03.2009,60,Hanumaiah,Vishal Bhardwaj
95980236-Rental-Agreement,9000,01.04.2010,28.02.2011,30,S.Sakunthala,V.V.Ravi Kian


## 2. Per-field recall on the test set
*Strict* = exact string match (the assignment metric). *Lenient* ignores letter case and extra spaces.

In [3]:
recall_test = per_field_recall(pred_test, labels)
recall_test.to_csv(ROOT / "outputs" / "recall_test.csv", index=False)
recall_test

,Field,Correct (strict),Recall (strict),Precision (strict),F1 (strict),Correct (lenient),Recall (lenient),Precision (lenient),F1 (lenient),Docs
0,Aggrement Value,4,1.000,1.000,1.000,4,1.000,1.000,1.000,4
1,Aggrement Start Date,4,1.000,1.000,1.000,4,1.000,1.000,1.000,4
2,Aggrement End Date,3,0.750,0.750,0.750,3,0.750,0.750,0.750,4
3,Renewal Notice (Days),4,1.000,1.000,1.000,4,1.000,1.000,1.000,4
4,Party One,4,1.000,1.000,1.000,4,1.000,1.000,1.000,4
5,Party Two,2,0.500,0.500,0.500,2,0.500,0.500,0.500,4
6,ALL FIELDS (micro),21,0.875,0.875,0.875,21,0.875,0.875,0.875,4


## 3. Validation on the labelled training files

In [4]:
pred_train = predict_folder(ROOT / "data" / "train", only=set(labels.index))
pred_train.to_csv(ROOT / "outputs" / "predictions_train.csv")
recall_train = per_field_recall(pred_train, labels)
recall_train.to_csv(ROOT / "outputs" / "recall_train.csv", index=False)
recall_train

Cached     18325926-Rental-Agreement-1.docx
Cached     36199312-Rental-Agreement.png
Cached     44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx
Cached     47854715-RENTAL-AGREEMENT.docx
Cached     50070534-RENTAL-AGREEMENT.docx
Cached     54770958-Rental-Agreement.png
Cached     54945838-Rental-Agreement.png
Cached     6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx
Cached     6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx


,Field,Correct (strict),Recall (strict),Precision (strict),F1 (strict),Correct (lenient),Recall (lenient),Precision (lenient),F1 (lenient),Docs
0,Aggrement Value,6,0.667,0.750,0.706,6,0.667,0.750,0.706,9
1,Aggrement Start Date,6,0.667,0.750,0.706,6,0.667,0.750,0.706,9
2,Aggrement End Date,3,0.333,0.375,0.353,3,0.333,0.375,0.353,9
3,Renewal Notice (Days),6,0.667,0.714,0.690,6,0.667,0.714,0.690,9
4,Party One,3,0.333,0.333,0.333,3,0.333,0.333,0.333,9
5,Party Two,5,0.556,0.556,0.556,5,0.556,0.556,0.556,9
6,ALL FIELDS (micro),29,0.537,0.571,0.554,29,0.537,0.571,0.554,9


## 4. Where the predictions differ from the labels

In [5]:
pd.concat([mismatches(pred_test, labels).assign(Split="test"),
           mismatches(pred_train, labels).assign(Split="train")], ignore_index=True)

,File,Field,Expected,Predicted,Split
0,156155545-Rental-Agreement-Kns-Home,Party Two,VYSHNAVI DAIRY SPECIALITIES Private Ltd,SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.,test
1,228094620-Rental-Agreement,Party Two,.B.Kishore,B.Kishore,test
2,95980236-Rental-Agreement,Aggrement End Date,31.03.2011,28.02.2011,test
3,18325926-Rental-Agreement-1,Aggrement End Date,31.11.2009,04.11.2009,train
4,18325926-Rental-Agreement-1,Party One,MR.K.Kuttan,K.Kuttan,train
5,36199312-Rental-Agreement,Aggrement End Date,31.04.2011,31.03.2011,train
6,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,Aggrement Value,3000,,train
7,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,Aggrement Start Date,20.09.2010,,train
8,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,Aggrement End Date,19.07.2011,,train
9,44737744-Maddireddy-Bhargava-Reddy-Rental-Agre...,Party One,M.V.V. VIJAYA SHANKAR,"M. V, V« VI JAY A SHANKAR",train


## 5. Using the REST API (optional)
Start the server in a terminal first: `uvicorn api.main:app --reload`

In [6]:
import requests
sample = ROOT / "data" / "test" / "24158401-Rental-Agreement.png"
try:
    with open(sample, "rb") as f:
        r = requests.post("http://127.0.0.1:8000/extract", files={"file": (sample.name, f)}, timeout=180)
    print(r.status_code)
    print(r.json())
except requests.exceptions.ConnectionError:
    print("API server is not running - start it with: uvicorn api.main:app --reload")

API server is not running - start it with: uvicorn api.main:app --reload
